<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/week3a2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Data Cleaning and Descriptive Statistics

In [1]:
import pandas as pd
import numpy as np

# Load original loans dataset with explicit numbers fully restored
loans = pd.DataFrame({
    'applicant_id': range(1, 15),
    'monthly_income': [180000, 220000, np.nan, 95000, 2400000, 150000, np.nan,
                       310000, 125000, np.nan, 205000, 170000, 140000, 260000],
    'loan_amount': [500000, 800000, 1200000, 300000, 2000000, 450000, 900000,
                    700000, 350000, 1500000, 600000, 400000, 380000, 850000],
    'employment_type': ['Salaried', None, 'Salaried', 'Self employed', 'Salaried',
                        None, 'Contract', 'Salaried', 'Contract', 'Salaried',
                        'Salaried', 'Contract', None, 'Salaried'],
    'referral_code': ['R12', None, None, None, 'R07', None, None,
                      None, None, 'R21', None, None, None, None],
    'defaulted': ['No', 'No', 'Yes', 'No', 'No', 'No', 'Yes',
                  'No', 'No', 'No', 'No', 'Yes', 'No', 'No'],
})

# --- PRO TIP FIX: Immediately capture the raw unmutated state ---
loans_raw = loans.copy()

print("Loans dataset successfully loaded without structural breaks!")


Loans dataset successfully loaded without structural breaks!


In [2]:
def missing_review(df, col, outcome):
    """
    Prints missing percentage for a column, and compares the rate and count
    of an outcome between rows where the column is missing vs present.
    """
    is_missing = df[col].isna()
    missing_pct = is_missing.mean() * 100

    print(f"=== MISSING REVIEW FOR COLUMN: {col} ===")
    print(f"Percent Missing: {missing_pct:.1f}%")

    summary = (df[outcome] == 'Yes').groupby(is_missing).agg(['mean', 'count'])
    summary.index.name = f"{col}_is_missing"
    summary.columns = [f'{outcome}_rate', 'group_size']

    # Format the rates as percentages for clearer terminal reading
    summary[f'{outcome}_rate'] = (summary[f'{outcome}_rate'] * 100).round(1)
    print(summary)
    print("\n" + "="*40 + "\n")

# Run directly on the unmutated tracking snapshot
missing_review(loans_raw, 'monthly_income', 'defaulted')
missing_review(loans_raw, 'employment_type', 'defaulted')


=== MISSING REVIEW FOR COLUMN: monthly_income ===
Percent Missing: 21.4%
                           defaulted_rate  group_size
monthly_income_is_missing                            
False                                 9.1          11
True                                 66.7           3


=== MISSING REVIEW FOR COLUMN: employment_type ===
Percent Missing: 21.4%
                            defaulted_rate  group_size
employment_type_is_missing                            
False                                 27.3          11
True                                   0.0           3




In [3]:
# --- 1. Stat Estimation ---
# Prediction: The mean will be significantly larger than the median. The value 2,400,000 is an extreme high outlier that will heavily pull the mean upward, whereas the median is robust to outliers.
# Chosen for filling: Median (180,000).
# Justification: The median preserves the central tendency of typical applicants without being distorted by the extreme income outlier.

print("--- 1. Income Mean vs Median ---")
print("Mean monthly income:", loans['monthly_income'].mean())
print("Median monthly income:", loans['monthly_income'].median())

# --- 2. Flag First ---
# Prediction: Applicants with missing income will have a higher default rate, as incomplete financial profiles often correlate with higher baseline credit risk.
loans['income_missing'] = loans['monthly_income'].isna()

print("\n--- 2. Default Rate & Sample Size by Income Missing Status ---")
income_risk_report = (loans['defaulted'] == 'Yes').groupby(loans['income_missing']).agg(['mean', 'count'])
income_risk_report.columns = ['default_rate', 'group_size']
display(income_risk_report)

# --- 3. Fill and Verify ---
loans['monthly_income'] = loans['monthly_income'].fillna(loans['monthly_income'].median())

print("\n--- 3. Post-Fill Verification ---")
print("Missing values remaining in monthly_income:", loans['monthly_income'].isna().sum())
print("True flags remaining in income_missing:", loans['income_missing'].sum())


--- 1. Income Mean vs Median ---
Mean monthly income: 386818.1818181818
Median monthly income: 180000.0

--- 2. Default Rate & Sample Size by Income Missing Status ---


,default_rate,group_size
income_missing,,
False,0.090909,11
True,0.666667,3



--- 3. Post-Fill Verification ---
Missing values remaining in monthly_income: 0
True flags remaining in income_missing: 3


In [4]:
# --- 1. Employment Type ---
# Choice: Fill with 'Unknown'
# Justification: Since 'Salaried', 'Self employed', and 'Contract' have distinctly different risk implications, forcing missing entries into the mode ('Salaried') could artificially skew risk models.

loans['employment_type'] = loans['employment_type'].fillna('Unknown')
print("--- 1. Employment Type Post-Fill Value Counts ---")
print(loans['employment_type'].value_counts(dropna=False))

# --- 2. Referral Code ---
# Context Query: A missing referral code most likely means the applicant was a direct organic sign-up and was not referred by anyone.
# Choice: Flag it (Convert to a boolean column indicating if they were referred).
# Justification: Transforming a sparsely populated string ID into a 'is_referred' flag preserves the behavioral signal (referred vs organic) while making it immediately useful for modeling.

loans['was_referred'] = loans['referral_code'].notna()
loans = loans.drop(columns=['referral_code'])

print("\n--- 2. Referral Code Transformation Output ---")
print(loans[['applicant_id', 'was_referred']].head())


--- 1. Employment Type Post-Fill Value Counts ---
employment_type
Salaried         7
Unknown          3
Contract         3
Self employed    1
Name: count, dtype: int64

--- 2. Referral Code Transformation Output ---
   applicant_id  was_referred
0             1          True
1             2         False
2             3         False
3             4         False
4             5          True


### Summary Analysis: Missing Value Behavioral Patterns (Loans Dataset)

By evaluating the relationship between missing attributes and default outcomes on our raw snapshot (`loans_raw`), we uncover critical risk signals that validate our feature-engineering choices.

#### 1. Monthly Income Missingness (Extreme-Risk Signal)
* **The Pattern:** Applicants with missing income values exhibit a **66.7% default rate** (2 out of 3), whereas applicants who reported their income exhibit only a **9.1% default rate** (1 out of 11).
* **Business Implication:** Missing income is strongly **Not Missing At Random (NMAR)**. Omitted financial parameters represent an acute credit flight risk in this dataset.
* **Data Strategy Validation:** This confirms why generating our boolean `income_missing` column *before* running imputation steps was vital. Imputing values without flagging would have completely blended this critical behavioral indicator into the normal sample variance.

#### 2. Employment Type Missingness (Low-Risk Baseline)
* **The Pattern:** Applicants with missing employment types show a **0.0% default rate** (0 out of 3), while identified profiles carry a **27.3% default rate** (3 out of 11).
* **Data Strategy Validation:** This completely supports our decision to impute missing rows with `'Unknown'` rather than forcing them into the modal category (`'Salaried'`). Artificially mapping these safe records to the mode would have introduced false risk configurations into our group baseline calculations.
